In [8]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Circle
from scipy.spatial import cKDTree
import math
import time

# ==================== ПАРАМЕТРЫ ====================
SIGMA = 1.0
R_CUT = 2.5 * SIGMA
EPSILON = 1.0

# --- Преграда ---
BARRIER_NY = 60
BARRIER_THICKNESS = 6
BARRIER_SPACING = 1.12 * SIGMA
BARRIER_X = 0.0
BARRIER_MASS = 50.0

# --- Цель ---
BODY_NX, BODY_NY = 5, 25
BODY_SPACING = 1.12 * SIGMA
BODY_OFFSET_X = 25.0
BODY_OFFSET_Y = 0.0
BODY_MASS = 3.0

# --- Снаряд (аналитический круг) ---
PROJECTILE_MASS = 20.0
PROJECTILE_RADIUS = 2.5
PROJECTILE_VELOCITY = 20.0

# --- Старт снаряда ---
USE_MANUAL_START = True
PROJECTILE_START_X = -50.0
PROJECTILE_START_Y = 60.0

# --- Угол (если USE_MANUAL_START = False) ---
IMPACT_ANGLE_DEG = -45.0

# --- Точка прицеливания ---
TARGET_X = BARRIER_X
TARGET_Y = 30.0

# --- Интегрирование ---
DT = 0.0005
N_STEPS = 20000
SAVE_EVERY = 10*5

# --- Пол ---
K_FLOOR = 8000.0
FLOOR_EPS = 0.0

# --- Контакт снаряда с частицами ---
PROJ_BARRIER_STIFFNESS = 15000.0
PROJ_BODY_STIFFNESS    = 8000.0
PROJ_BARRIER_LJ_SCALE  = 0.4
PROJ_BODY_LJ_SCALE     = 0.15

# --- Критерий пробоя ---
BREACH_PATH_HALF_WIDTH = 4.0
BREACH_MIN_PARTICLES = 3

# --- Критерий поражения цели ---
HIT_RADIUS = 0.9 * SIGMA
HIT_SPEED_THRESHOLD = 3.0
HIT_IMPULSE_THRESHOLD = 30.0


# ==================== ИНИЦИАЛИЗАЦИЯ ====================
def create_barrier():
    positions = []
    for i in range(BARRIER_THICKNESS):
        for j in range(BARRIER_NY):
            x = BARRIER_X + i * BARRIER_SPACING
            y = j * BARRIER_SPACING
            positions.append([x, y])
    return np.array(positions)


def create_body():
    positions = []
    for i in range(BODY_NX):
        for j in range(BODY_NY):
            x = BODY_OFFSET_X + i * BODY_SPACING
            y = BODY_OFFSET_Y + j * BODY_SPACING
            positions.append([x, y])
    return np.array(positions)


def create_projectile():
    """Снаряд — одна точка с массой и радиусом."""
    if USE_MANUAL_START:
        x0, y0 = PROJECTILE_START_X, PROJECTILE_START_Y
        dx = TARGET_X - x0
        dy = TARGET_Y - y0
        norm = math.hypot(dx, dy)
        dir_x, dir_y = dx / norm, dy / norm
        angle_deg = math.degrees(math.atan2(dy, dx))
        print(f"Угол полёта снаряда: {angle_deg:.1f}°")
    else:
        angle_rad = math.radians(IMPACT_ANGLE_DEG)
        dir_x, dir_y = math.cos(angle_rad), math.sin(angle_rad)
        start_distance = 12.0
        x0 = TARGET_X - dir_x * start_distance
        y0 = TARGET_Y - dir_y * start_distance

    pos = np.array([[x0, y0]])
    vel = np.array([[dir_x * PROJECTILE_VELOCITY,
                     dir_y * PROJECTILE_VELOCITY]])
    mass = np.array([PROJECTILE_MASS])
    return pos, vel, mass


# ==================== СИЛЫ ====================
def compute_pair_forces(positions, pairs):
    i, j = pairs[:, 0], pairs[:, 1]
    dr = positions[j] - positions[i]
    r = np.linalg.norm(dr, axis=1)
    r = np.maximum(r, 1e-6)

    sr = SIGMA / r
    sr6 = sr ** 6
    sr12 = sr6 ** 2
    f_mag = 24.0 * EPSILON * (2.0 * sr12 - sr6) / r
    f_mag = np.clip(f_mag, -8000.0, 8000.0)

    f_vec = f_mag[:, None] * dr / r[:, None]
    return i, j, f_vec


def compute_forces(positions, tree, is_barrier, is_body):
    forces = np.zeros_like(positions)

    pairs = tree.query_pairs(R_CUT, output_type='ndarray')
    if len(pairs) > 0:
        i, j, f_vec = compute_pair_forces(positions, pairs)
        np.add.at(forces, i, -f_vec)
        np.add.at(forces, j, f_vec)

    # Пол
    y = positions[:, 1]
    below = y < FLOOR_EPS
    if np.any(below):
        forces[below, 1] += K_FLOOR * (FLOOR_EPS - y[below])

    return forces


def compute_projectile_force(p_pos, positions, is_barrier, is_body):
    """Сила, действующая на снаряд со стороны всех частиц."""
    total_f = np.zeros(2)

    for idx_group, stiffness, lj_scale in [
        (np.where(is_barrier)[0], PROJ_BARRIER_STIFFNESS,
         PROJ_BARRIER_LJ_SCALE),
        (np.where(is_body)[0], PROJ_BODY_STIFFNESS,
         PROJ_BODY_LJ_SCALE),
    ]:
        if len(idx_group) == 0:
            continue

        d_vecs = positions[idx_group] - p_pos
        ds = np.linalg.norm(d_vecs, axis=1)
        ds = np.maximum(ds, 1e-9)

        # Внутри круга — выталкивание наружу
        inside = ds < PROJECTILE_RADIUS
        if np.any(inside):
            f_mag = stiffness * (PROJECTILE_RADIUS - ds[inside])
            unit = d_vecs[inside] / ds[inside, None]
            total_f -= np.sum(f_mag[:, None] * unit, axis=0)

        # На границе — LJ
        boundary = (ds >= PROJECTILE_RADIUS) & (ds < PROJECTILE_RADIUS + SIGMA)
        if np.any(boundary):
            r_eff = ds[boundary] - PROJECTILE_RADIUS + SIGMA
            sr = SIGMA / r_eff
            sr6 = sr ** 6
            sr12 = sr6 ** 2
            f_lj = 24.0 * EPSILON * (2.0 * sr12 - sr6) / r_eff
            f_lj = np.clip(f_lj, -8000.0, 8000.0) * lj_scale
            unit = d_vecs[boundary] / ds[boundary, None]
            total_f -= np.sum(f_lj[:, None] * unit, axis=0)

    # Пол для снаряда
    if p_pos[1] < FLOOR_EPS + PROJECTILE_RADIUS:
        overlap = (FLOOR_EPS + PROJECTILE_RADIUS) - p_pos[1]
        total_f[1] += K_FLOOR * overlap

    return total_f


def apply_projectile_forces_to_particles(p_pos, positions, forces,
                                         is_barrier, is_body):
    """Силы от снаряда на частицы преграды и цели."""
    for idx_group, stiffness, lj_scale in [
        (np.where(is_barrier)[0], PROJ_BARRIER_STIFFNESS,
         PROJ_BARRIER_LJ_SCALE),
        (np.where(is_body)[0], PROJ_BODY_STIFFNESS,
         PROJ_BODY_LJ_SCALE),
    ]:
        if len(idx_group) == 0:
            continue

        d_vecs = positions[idx_group] - p_pos
        ds = np.linalg.norm(d_vecs, axis=1)
        ds = np.maximum(ds, 1e-9)

        inside = ds < PROJECTILE_RADIUS
        if np.any(inside):
            f_mag = stiffness * (PROJECTILE_RADIUS - ds[inside])
            unit = d_vecs[inside] / ds[inside, None]
            forces[idx_group[inside]] += f_mag[:, None] * unit

        boundary = (ds >= PROJECTILE_RADIUS) & (ds < PROJECTILE_RADIUS + SIGMA)
        if np.any(boundary):
            r_eff = ds[boundary] - PROJECTILE_RADIUS + SIGMA
            sr = SIGMA / r_eff
            sr6 = sr ** 6
            sr12 = sr6 ** 2
            f_lj = 24.0 * EPSILON * (2.0 * sr12 - sr6) / r_eff
            f_lj = np.clip(f_lj, -8000.0, 8000.0) * lj_scale
            unit = d_vecs[boundary] / ds[boundary, None]
            forces[idx_group[boundary]] += f_lj[:, None] * unit


def barrier_breached(positions, is_barrier, proj_pos):
    barrier_pos = positions[is_barrier]
    if len(barrier_pos) == 0:
        return True

    x_min = BARRIER_X - 0.5
    x_max = BARRIER_X + BARRIER_THICKNESS * BARRIER_SPACING + 0.5
    in_slab = (barrier_pos[:, 0] >= x_min) & (barrier_pos[:, 0] <= x_max)

    y_proj = proj_pos[1]
    near_path = in_slab & (np.abs(barrier_pos[:, 1] - y_proj)
                           < BREACH_PATH_HALF_WIDTH)

    return int(np.sum(near_path)) < BREACH_MIN_PARTICLES


def check_body_hit(positions, velocities, proj_pos, proj_vel,
                   is_body, is_barrier, masses, step=None):
    """Возвращает (hit, who, info). who: 'proj', 'shrapnel' или None."""
    body_idx = np.where(is_body)[0]
    if len(body_idx) == 0:
        return False, None, None

    body_pos = positions[body_idx]
    body_vel = velocities[body_idx]

    # --- Проверка снаряда ---
    for k, bp in enumerate(body_pos):
        d = np.linalg.norm(proj_pos - bp)
        if d < HIT_RADIUS + PROJECTILE_RADIUS:
            rel_v = np.linalg.norm(proj_vel - body_vel[k])
            impulse = PROJECTILE_MASS * rel_v
            if rel_v > HIT_SPEED_THRESHOLD and impulse > HIT_IMPULSE_THRESHOLD:
                info = (f"СНАРЯДОМ: d={d:.2f}, rel_v={rel_v:.1f}, "
                        f"imp={impulse:.0f}, снаряд={proj_pos}, цель={bp}")
                print(f"[ШАГ {step}] ПОРАЖЕНИЕ {info}")
                return True, 'proj', info

    # --- Проверка осколков ---
    barrier_idx = np.where(is_barrier)[0]
    if len(barrier_idx) == 0:
        return False, None, None

    att_pos = positions[barrier_idx]
    att_vel = velocities[barrier_idx]
    att_mass = masses[barrier_idx]

    tree = cKDTree(att_pos)
    for k, bp in enumerate(body_pos):
        idxs = tree.query_ball_point(bp, HIT_RADIUS + 0.5)
        for ai in idxs:
            d = np.linalg.norm(att_pos[ai] - bp)
            if d > HIT_RADIUS:
                continue
            rel_v = np.linalg.norm(att_vel[ai] - body_vel[k])
            impulse = att_mass[ai] * rel_v
            if rel_v > HIT_SPEED_THRESHOLD and impulse > HIT_IMPULSE_THRESHOLD:
                info = (f"ОСКОЛКОМ: d={d:.2f}, rel_v={rel_v:.1f}, "
                        f"imp={impulse:.0f}, осколок={att_pos[ai]}, "
                        f"v_оск={att_vel[ai]}, цель={bp}, v_цели={body_vel[k]}")
                print(f"[ШАГ {step}] ПОРАЖЕНИЕ {info}")
                return True, 'shrapnel', info

    return False, None, None


# ==================== СИМУЛЯЦИЯ ====================
def simulate():
    barrier_pos = create_barrier()
    body_pos = create_body()
    proj_pos_arr, proj_vel, proj_mass = create_projectile()

    n_barrier = len(barrier_pos)
    n_body = len(body_pos)

    positions = np.vstack([barrier_pos, body_pos])
    velocities = np.vstack([np.zeros_like(barrier_pos),
                            np.zeros_like(body_pos)])
    masses = np.concatenate([np.full(n_barrier, BARRIER_MASS),
                             np.full(n_body, BODY_MASS)])

    is_barrier = np.array([True] * n_barrier + [False] * n_body)
    is_body = np.array([False] * n_barrier + [True] * n_body)

    proj_pos = proj_pos_arr[0].copy()
    proj_vel = proj_vel[0].copy()

    print(f"Частиц: {len(positions)} "
          f"(преграда: {n_barrier}, цель: {n_body})")
    print(f"Снаряд: масса {PROJECTILE_MASS}, радиус {PROJECTILE_RADIUS}, "
          f"скорость {PROJECTILE_VELOCITY}")
    print(f"Преграда: масса частицы {BARRIER_MASS}")

    tree = cKDTree(positions)
    forces = compute_forces(positions, tree, is_barrier, is_body)
    proj_force = compute_projectile_force(proj_pos, positions,
                                          is_barrier, is_body)
    apply_projectile_forces_to_particles(proj_pos, positions, forces,
                                         is_barrier, is_body)

    history_particles = []
    history_proj = []

    t_start = time.time()
    breach_step = None
    hit_step = None
    hit_who = None

    for step in range(N_STEPS):
        # --- Снаряд: полушаг ---
        proj_vel += 0.5 * proj_force / PROJECTILE_MASS * DT
        proj_pos += proj_vel * DT

        if proj_pos[1] < FLOOR_EPS + PROJECTILE_RADIUS:
            proj_pos[1] = FLOOR_EPS + PROJECTILE_RADIUS
            proj_vel[1] = max(proj_vel[1], 0.0)

        # --- Частицы: полушаг ---
        velocities += 0.5 * forces / masses[:, None] * DT
        positions += velocities * DT

        below = positions[:, 1] < FLOOR_EPS
        if np.any(below):
            positions[below, 1] = FLOOR_EPS
            velocities[below, 1] = np.maximum(velocities[below, 1], 0.0)

        # --- Проверка пробоя ---
        if breach_step is None:
            if barrier_breached(positions, is_barrier, proj_pos):
                breach_step = step
                print(f"*** ПРЕГРАДА ПРОБИТА на шаге {step} *** "
                      f"снаряд={proj_pos}, "
                      f"|v|={np.linalg.norm(proj_vel):.1f}")

        # --- Проверка поражения ---
        if hit_step is None:
            hit, who, info = check_body_hit(
                positions, velocities, proj_pos, proj_vel,
                is_body, is_barrier, masses, step=step)
            if hit:
                hit_step = step
                hit_who = who
                print(f"*** ЦЕЛЬ ПОРАЖЕНА на шаге {step} "
                      f"({who}) ***")

        # --- Пересчёт сил ---
        tree = cKDTree(positions)
        forces = compute_forces(positions, tree, is_barrier, is_body)
        proj_force = compute_projectile_force(proj_pos, positions,
                                              is_barrier, is_body)
        apply_projectile_forces_to_particles(proj_pos, positions, forces,
                                             is_barrier, is_body)

        # --- Второй полушаг ---
        proj_vel += 0.5 * proj_force / PROJECTILE_MASS * DT
        velocities += 0.5 * forces / masses[:, None] * DT

        if step % SAVE_EVERY == 0:
            history_particles.append(positions.copy())
            history_proj.append(proj_pos.copy())

        if step % 2000 == 0:
            elapsed = time.time() - t_start
            if step > 0:
                eta = elapsed / step * (N_STEPS - step)
                print(f"Шаг {step}/{N_STEPS} | {elapsed:.1f} с | "
                      f"~{eta:.0f} с осталось")
            else:
                print(f"Шаг {step}/{N_STEPS}")

    print(f"Готово за {time.time() - t_start:.1f} с. Кадров: "
          f"{len(history_particles)}")
    if breach_step is not None:
        print(f"Пробой: шаг {breach_step}")
    if hit_step is not None:
        print(f"Поражение цели: шаг {hit_step} ({hit_who})")
    else:
        print("Цель не поражена за отведённое время.")

    return (history_particles, history_proj,
            is_barrier, is_body, breach_step, hit_step, hit_who)


# ==================== GIF ====================
def make_gif(history_particles, history_proj,
             is_barrier, is_body, breach_step, hit_step, hit_who,
             filename='penetration.gif'):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect('equal')

    all_pos = np.vstack(history_particles)
    all_proj = np.array(history_proj)
    margin = 3.0

    x_min = min(all_pos[:, 0].min(), all_proj[:, 0].min()) - margin
    x_max = max(all_pos[:, 0].max(), all_proj[:, 0].max()) + margin
    y_min = min(0.0, all_pos[:, 1].min(), all_proj[:, 1].min()) - 1.0
    y_max = max(all_pos[:, 1].max(), all_proj[:, 1].max()) + margin
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_facecolor('black')
    fig.patch.set_facecolor('black')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_color('white')

    ax.axhline(0.0, color='gray', linewidth=1.5,
               linestyle='--', label='Пол (y=0)')

    barrier_scatter = ax.scatter([], [], s=25, c='orange', label='Преграда')
    body_scatter = ax.scatter([], [], s=25, c='cyan', label='Цель')

    proj_circle = Circle((0, 0), PROJECTILE_RADIUS, color='red',
                         zorder=10, label='Снаряд')
    ax.add_patch(proj_circle)

    title = ax.set_title('', color='white')
    ax.legend(loc='upper right', facecolor='black', labelcolor='white')

    breach_frame = (breach_step // SAVE_EVERY) if breach_step is not None else None
    hit_frame = (hit_step // SAVE_EVERY) if hit_step is not None else None

    def update(frame):
        pos = history_particles[frame]
        proj = history_proj[frame]

        barrier_scatter.set_offsets(pos[is_barrier])
        body_scatter.set_offsets(pos[is_body])

        proj_circle.center = (proj[0], proj[1])

        # Цвет цели: голубая → жёлтая (поражена осколком) → красная (снарядом)
        if hit_frame is not None and frame >= hit_frame:
            if hit_who == 'proj':
                body_scatter.set_color('red')
            else:
                body_scatter.set_color('yellow')
        else:
            body_scatter.set_color('cyan')

        status = f'Кадр {frame}/{len(history_particles) - 1}'
        if breach_frame is not None and frame >= breach_frame:
            status += ' | преграда пробита'
        if hit_frame is not None and frame >= hit_frame:
            if hit_who == 'proj':
                status += ' | ЦЕЛЬ ПОРАЖЕНА СНАРЯДОМ'
            else:
                status += ' | ЦЕЛЬ ПОРАЖЕНА ОСКОЛКОМ'
        title.set_text(status)

        return barrier_scatter, body_scatter, proj_circle, title

    ani = FuncAnimation(fig, update, frames=len(history_particles),
                        interval=50, blit=False, repeat=False)

    print(f"Сохранение GIF ({len(history_particles)} кадров)...")
    writer = PillowWriter(fps=25)
    ani.save(filename, writer=writer, dpi=80)
    print(f"Готово! Файл: {filename}")
    plt.close(fig)


# ==================== ЗАПУСК ====================
if __name__ == "__main__":
    print("Моделирование: снаряд-круг → преграда → цель")
    print("Снаряд монолитный (не распадается). Осколки преграды "
          "могут поразить цель.")
    (history_particles, history_proj, is_barrier, is_body,
     breach_step, hit_step, hit_who) = simulate()
    print("Создание GIF...")
    make_gif(history_particles, history_proj,
             is_barrier, is_body, breach_step, hit_step, hit_who,
             filename='penetration.gif')

Моделирование: снаряд-круг → преграда → цель
Снаряд монолитный (не распадается). Осколки преграды могут поразить цель.
Угол полёта снаряда: -31.0°
Частиц: 485 (преграда: 360, цель: 125)
Снаряд: масса 20.0, радиус 2.5, скорость 20.0
Преграда: масса частицы 50.0
Шаг 0/20000
Шаг 2000/20000 | 5.0 с | ~45 с осталось
Шаг 4000/20000 | 10.4 с | ~41 с осталось
Шаг 6000/20000 | 15.8 с | ~37 с осталось
Шаг 8000/20000 | 20.9 с | ~31 с осталось
Шаг 10000/20000 | 26.1 с | ~26 с осталось
Шаг 12000/20000 | 31.3 с | ~21 с осталось
Шаг 14000/20000 | 36.6 с | ~16 с осталось
Шаг 16000/20000 | 41.9 с | ~10 с осталось
Шаг 18000/20000 | 47.1 с | ~5 с осталось
Готово за 52.3 с. Кадров: 400
Цель не поражена за отведённое время.
Создание GIF...
Сохранение GIF (400 кадров)...
Готово! Файл: penetration.gif
